# 04 — SAR Change Detection: Can You See the Damage?

**Goal:** Compute a pre/post backscatter change image and extract per-parcel
zonal statistics to quantify fire damage from SAR.

The change image is simply: **change = post_dB - pre_dB**

- **Negative values** = backscatter decreased (buildings destroyed, vegetation burned)
- **Positive values** = backscatter increased (debris piles, regrowth)
- **Near zero** = no significant change

In [ ]:
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import planetary_computer
import pystac_client
import rasterio
from rasterio.windows import from_bounds

try:
    from rasterstats import zonal_stats
    HAS_RASTERSTATS = True
except ImportError:
    HAS_RASTERSTATS = False
    print("rasterstats not installed -- zonal stats cells will be skipped.")
    print("Install with: pip install rasterstats")

%matplotlib inline
plt.rcParams["figure.dpi"] = 120

In [ ]:
# ---------------------------------------------------------------------------
# Load pre and post VV via Planetary Computer, convert to dB, compute change
# ---------------------------------------------------------------------------

AOI_BBOX = [-105.16, 39.93, -105.07, 40.01]


def linear_to_db(dn_array):
    """Convert raw DN (linear amplitude) to sigma0 in decibels."""
    power = dn_array.astype(np.float64) ** 2
    power = np.where(power > 0, power, 1e-30)
    return (10.0 * np.log10(power)).astype(np.float32)


def load_vv_chip(item, bbox):
    """Read the VV band windowed to bbox and return (array, transform, crs)."""
    with rasterio.open(item.assets["vv"].href) as src:
        window = from_bounds(*bbox, transform=src.transform)
        data = src.read(1, window=window).astype(np.float32)
        win_transform = src.window_transform(window)
        crs = src.crs
    return data, win_transform, crs


# Connect to STAC
catalog = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1",
    modifier=planetary_computer.sign_inplace,
)

# Pre-fire (Nov 2021)
items_pre = list(catalog.search(
    collections=["sentinel-1-grd"], bbox=AOI_BBOX,
    datetime="2021-11-01/2021-11-30",
).items())

# Post-fire (Jan 2022)
items_post = list(catalog.search(
    collections=["sentinel-1-grd"], bbox=AOI_BBOX,
    datetime="2022-01-01/2022-01-31",
).items())

item_pre = items_pre[0]
item_post = items_post[0]

print(f"Pre-fire:  {item_pre.id}  ({item_pre.datetime.date()})")
print(f"Post-fire: {item_post.id} ({item_post.datetime.date()})")

# Load and calibrate
vv_pre_raw, transform_pre, crs = load_vv_chip(item_pre, AOI_BBOX)
vv_post_raw, transform_post, _ = load_vv_chip(item_post, AOI_BBOX)

vv_pre_db = linear_to_db(vv_pre_raw)
vv_post_db = linear_to_db(vv_post_raw)

# Handle shape mismatch -- crop to the smaller extent
min_rows = min(vv_pre_db.shape[0], vv_post_db.shape[0])
min_cols = min(vv_pre_db.shape[1], vv_post_db.shape[1])
vv_pre_db = vv_pre_db[:min_rows, :min_cols]
vv_post_db = vv_post_db[:min_rows, :min_cols]

# Compute change
vv_change = vv_post_db - vv_pre_db

print(f"\nChip shape: {vv_change.shape}")
print(f"Change range: {np.nanmin(vv_change):.2f} to {np.nanmax(vv_change):.2f} dB")
print(f"Change mean:  {np.nanmean(vv_change):.2f} dB")
print(f"Change std:   {np.nanstd(vv_change):.2f} dB")

## Visualize change image -- negative = backscatter loss

We use a diverging colormap (RdBu) so that:
- **Red** = negative change (backscatter decreased = likely damage)
- **Blue** = positive change (backscatter increased)
- **White** = no change

In [ ]:
# ---------------------------------------------------------------------------
# Plot the VV change image
# ---------------------------------------------------------------------------

fig, ax = plt.subplots(figsize=(10, 10))
im = ax.imshow(vv_change, cmap="RdBu", vmin=-8, vmax=8)
ax.set_title(f"VV Change (post - pre) dB\n"
             f"{item_post.datetime.date()} minus {item_pre.datetime.date()}",
             fontsize=13)
ax.set_xlabel("Column")
ax.set_ylabel("Row")
cbar = plt.colorbar(im, ax=ax, shrink=0.7, label="dB change")
plt.tight_layout()
plt.show()

## Overlay parcel boundaries if available

If a parcels GeoJSON file exists in the data directory, we overlay the
boundaries on the change map. This helps visualize which parcels fall in
the damaged zone.

In [ ]:
# ---------------------------------------------------------------------------
# Try to load and overlay parcel boundaries
# ---------------------------------------------------------------------------

parcels_gdf = None

# Check several possible locations for the parcels file
candidate_paths = [
    Path("../data/raw/parcels.geojson"),
    Path("../data/parcels.geojson"),
    Path("../data/raw/boulder_parcels.geojson"),
    Path("../data/processed/parcels.geojson"),
]

for p in candidate_paths:
    if p.exists():
        parcels_gdf = gpd.read_file(p)
        print(f"Loaded parcels from {p}: {len(parcels_gdf)} features")
        break

if parcels_gdf is not None:
    # Reproject parcels to match the raster CRS if needed
    if parcels_gdf.crs and parcels_gdf.crs != crs:
        parcels_gdf = parcels_gdf.to_crs(crs)

    # Create a georeferenced plot
    fig, ax = plt.subplots(figsize=(10, 10))
    extent = [
        transform_pre.c,
        transform_pre.c + transform_pre.a * min_cols,
        transform_pre.f + transform_pre.e * min_rows,
        transform_pre.f,
    ]
    ax.imshow(vv_change, cmap="RdBu", vmin=-8, vmax=8, extent=extent)
    parcels_gdf.boundary.plot(ax=ax, edgecolor="black", linewidth=0.5)
    ax.set_title("VV Change with Parcel Boundaries")
    plt.tight_layout()
    plt.show()
else:
    print("No parcels file found. Skipping overlay.")
    print("Expected locations:")
    for p in candidate_paths:
        print(f"  {p.resolve()}")

## Compute per-parcel zonal statistics

Using `rasterstats.zonal_stats`, we compute the mean VV change for each
parcel polygon. This produces the key feature for damage classification:
parcels with strongly negative mean change are likely damaged.

In [ ]:
# ---------------------------------------------------------------------------
# Compute per-parcel zonal statistics on the VV change image
# ---------------------------------------------------------------------------

if parcels_gdf is not None and HAS_RASTERSTATS:
    stats = zonal_stats(
        parcels_gdf,
        vv_change,
        affine=transform_pre,
        stats=["mean", "std", "min", "max", "count"],
        nodata=np.nan,
    )

    # Add results to GeoDataFrame
    stats_df = pd.DataFrame(stats)
    parcels_gdf["vv_change_mean"] = stats_df["mean"]
    parcels_gdf["vv_change_std"] = stats_df["std"]
    parcels_gdf["vv_change_min"] = stats_df["min"]
    parcels_gdf["vv_change_max"] = stats_df["max"]
    parcels_gdf["vv_pixel_count"] = stats_df["count"]

    print(f"Computed zonal stats for {len(parcels_gdf)} parcels.")
    print()
    print(parcels_gdf[["vv_change_mean", "vv_change_std",
                       "vv_change_min", "vv_change_max",
                       "vv_pixel_count"]].describe().round(3))
else:
    if not HAS_RASTERSTATS:
        print("Skipped: rasterstats is not installed.")
    else:
        print("Skipped: no parcels GeoDataFrame available.")
    print("\nCreating a synthetic demo instead so the histogram cell still works...")

    # Synthetic fallback for demonstration
    np.random.seed(42)
    n_synth = 1000
    parcels_gdf = gpd.GeoDataFrame({
        "vv_change_mean": np.concatenate([
            np.random.normal(-5.0, 1.5, int(n_synth * 0.3)),   # damaged
            np.random.normal(-0.5, 1.5, int(n_synth * 0.7)),   # undamaged
        ])
    })
    print(f"Created {len(parcels_gdf)} synthetic parcels for demonstration.")

## Distribution of VV change by damage class

The histogram shows the distribution of per-parcel mean VV change.
We draw threshold lines at **-2.0 dB** (moderate damage) and **-4.0 dB**
(severe damage). Parcels below these thresholds are more likely to have
suffered structural damage.

In [ ]:
# ---------------------------------------------------------------------------
# Histogram of per-parcel VV change with damage thresholds
# ---------------------------------------------------------------------------

values = parcels_gdf["vv_change_mean"].dropna()

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(values, bins=60, color="steelblue", edgecolor="white", alpha=0.8)

# Threshold lines
ax.axvline(x=-2.0, color="orange", linestyle="--", linewidth=2,
           label="Moderate damage threshold (-2.0 dB)")
ax.axvline(x=-4.0, color="red", linestyle="--", linewidth=2,
           label="Severe damage threshold (-4.0 dB)")

ax.set_xlabel("Mean VV change (dB)", fontsize=12)
ax.set_ylabel("Number of parcels", fontsize=12)
ax.set_title("Distribution of Per-Parcel VV Backscatter Change", fontsize=13)
ax.legend(fontsize=10)

# Annotate counts
n_moderate = (values < -2.0).sum()
n_severe = (values < -4.0).sum()
ax.text(0.02, 0.95,
        f"Below -2.0 dB: {n_moderate} parcels ({n_moderate/len(values)*100:.1f}%)\n"
        f"Below -4.0 dB: {n_severe} parcels ({n_severe/len(values)*100:.1f}%)",
        transform=ax.transAxes, fontsize=10, verticalalignment="top",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))

plt.tight_layout()
plt.show()

print("\nSummary:")
print(f"  Total parcels:        {len(values)}")
print(f"  Below -2.0 dB:        {n_moderate} ({n_moderate/len(values)*100:.1f}%)")
print(f"  Below -4.0 dB:        {n_severe} ({n_severe/len(values)*100:.1f}%)")